## Deep Dive into Transformers, Quantization & Neural Netwoek

In [2]:
# !pip install --upgrade bitsandbytes accelerate transformers==4.57.6


In [3]:
# libraries
from dotenv import load_dotenv
from huggingface_hub import login
import torch
from transformers import BitsAndBytesConfig, AutoTokenizer, AutoModelForCausalLM, TextStreamer
from IPython.display import display

### Working with HuggingFace Transformers Low-level API and Quantization

In [ ]:
HF_TOKEN='HUGGING FACE TOKEN'

load_dotenv(override=True)
# api_key = os.getenv('HF_TOKEN')

if not HF_TOKEN:
    print("API Keys not found")
else:
    print("API Key found and in use")

login(token=HF_TOKEN)
print('Login Successful')

API Key found and in use
Login Successful


In [5]:
LLAMA = 'meta-llama/Llama-3.2-1B-Instruct'

# LLAMA = 'meta-llama/Llama-3.1-8B-Instruct'
PHI = 'microsoft/Phi-4-mini-instruct'
GEMMA = 'google/gemma-3-270m-it'
QWEN = 'Qwen3-4B-Instruct-2507'
DEEPSEEK = 'deepseek-ai/DeepSeek-RI-Distill-Qwen-1.5B'

messages = [
    {"role": "user", "content": "Tell a joke for a room of Data Scientists"}
]

In [6]:
# Quantization Config to allow us to load the model into memory and use less memory

quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_quant_type='nf4'
)

In [7]:
tokenizer = AutoTokenizer.from_pretrained(LLAMA)
tokenizer.pad_token = tokenizer.eos_token
inputs = tokenizer.apply_chat_template(messages, return_tensors='pt').to('cuda')

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


In [8]:
# the model
model = AutoModelForCausalLM.from_pretrained(LLAMA, device_map='auto', quantization_config=quant_config)

In [9]:
memory = model.get_memory_footprint() / 1e6
print(f"Memory footprints: {memory:,.1f} MB")

Memory footprints: 1,012.0 MB


In [10]:
model

LlamaForCausalLM(
  (model): LlamaModel(
    (embed_tokens): Embedding(128256, 2048)
    (layers): ModuleList(
      (0-15): 16 x LlamaDecoderLayer(
        (self_attn): LlamaAttention(
          (q_proj): Linear4bit(in_features=2048, out_features=2048, bias=False)
          (k_proj): Linear4bit(in_features=2048, out_features=512, bias=False)
          (v_proj): Linear4bit(in_features=2048, out_features=512, bias=False)
          (o_proj): Linear4bit(in_features=2048, out_features=2048, bias=False)
        )
        (mlp): LlamaMLP(
          (gate_proj): Linear4bit(in_features=2048, out_features=8192, bias=False)
          (up_proj): Linear4bit(in_features=2048, out_features=8192, bias=False)
          (down_proj): Linear4bit(in_features=8192, out_features=2048, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): LlamaRMSNorm((2048,), eps=1e-05)
        (post_attention_layernorm): LlamaRMSNorm((2048,), eps=1e-05)
      )
    )
    (norm): LlamaRMSNorm

In [13]:
# running the model
outputs = model.generate(inputs, max_new_tokens=80)

# outputs = model.generate(input_ids=inputs['input_ids'], attention_mask=inputs['attention_mask'], max_new_tokens=80)
outputs[0]

outputs[0]

The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


tensor([128000, 128006,   9125, 128007,    271,  38766,   1303,  33025,   2696,
            25,   6790,    220,   2366,     18,    198,  15724,   2696,     25,
           220,    868,  17907,    220,   2366,     21,    271, 128009, 128006,
           882, 128007,    271,  41551,    264,  22380,    369,    264,   3130,
           315,   2956,  57116, 128009, 128006,  78191, 128007,    271,   8586,
           596,    264,  22380,  41891,    311,    264,   3130,    315,    828,
         14248,   1473,  10445,   1550,    279,    828,  28568,   1464,    709,
           449,    813,  23601,   1980,  18433,    568,   4934,    311,  24564,
           872,   5133,    323,   1505,    279,  26670,   1990,    872,  21958,
           323,    279,    828,     13,    320,  28548,    696,     40,   3987,
           430,    832,    364,  44803,  16284,      6,   1664,    323,   7263,
           264,  15648,    311,    872,  12580,      0, 128009],
       device='cuda:0')

In [14]:
tokenizer.decode(outputs[0])

"<|begin_of_text|><|start_header_id|>system<|end_header_id|>\n\nCutting Knowledge Date: December 2023\nToday Date: 15 Sep 2026\n\n<|eot_id|><|start_header_id|>user<|end_header_id|>\n\nTell a joke for a room of Data Scientists<|eot_id|><|start_header_id|>assistant<|end_header_id|>\n\nHere's a joke tailored to a room of data scientists:\n\nWhy did the data scientist break up with his girlfriend?\n\nBecause he wanted to analyze their relationship and find the correlation between their emotions and the data. (pause)\n\nI hope that one 'analyzed' well and brought a smile to their faces!<|eot_id|>"

In [ ]:
# cleaning up the memory

# import gc


# del model, inputs, tokenizer, outputs
# gc.collect()
# torch.cuda.empty_cache()

In [21]:
# Wrapping everything in a function - and adding Streaming and generation prompts

def generate_(model, messages, quant=True, max_new_tokens=80):
    tokenizer = AutoTokenizer.from_pretrained(model)
    tokenizer.pad_token = tokenizer.eos_token

    input_ids = tokenizer.apply_chat_template(
        messages,
        return_tensors="pt",
        add_generation_prompt=True
    ).to("cuda")

    attention_mask = torch.ones_like(
        input_ids,
        dtype=torch.long,
        device="cuda"
    )

    streamer = TextStreamer(tokenizer)

    if quant:
        model = AutoModelForCausalLM.from_pretrained(
            model,
            quantization_config=quant_config
        ).to("cuda")
    else:
        model = AutoModelForCausalLM.from_pretrained(
            model
        ).to("cuda")

    outputs = model.generate(
        input_ids=input_ids,
        attention_mask=attention_mask,
        max_new_tokens=max_new_tokens,
        streamer=streamer
    )

    return outputs

In [22]:
generate_(PHI, messages)

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/168 [00:00<?, ?B/s]

<|user|>Tell a joke for a room of Data Scientists<|end|><|assistant|>Sure, here's a joke tailored for data scientists:

Why did the data scientist break up with the computer?

Because it kept asking for more space, and she needed room for her own data!<|end|>


tensor([[200021,  60751,    261,  41751,    395,    261,   3435,    328,   4833,
         103481, 200020, 200019,  62915,     11,  60623,    261,  41751,  45089,
            395,   1238,  27356,   1402,  13903,   2242,    290,   1238,  57204,
           2338,    869,    483,    290,   7595,   1715,  30105,    480,  13185,
          16054,    395,    945,   4918,     11,    326,   1770,   6118,   3435,
            395,   1335,   2316,   1238,      0, 200020]], device='cuda:0')